In [ ]:
import pandas as pd

In [ ]:
# To access files in your Google Drive from Google Colaboratory, you need to authenticate your notebook with Google Drive. Run the following code to authenticate:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# read in the file - which in this case is in csv - comma separated value - format
from pathlib import PurePath

data_path = PurePath('/content/drive/My Drive/Colab Notebooks/AFL/AFL Fantasy Data')

In [ ]:

filename = 'afl_fantasy_league_test_data.csv'
afl_df = pd.read_csv(data_path / filename)

In [ ]:
afl_df

In [ ]:
# First 20 data points
# Use context manager to temporarily allow all columns to be shown
with pd.option_context('display.max_columns', None):
    display(afl_df.head(20))

In [ ]:
with pd.option_context('display.max_columns', None):
    display(afl_df.describe())

In [ ]:
with pd.option_context('display.max_columns', None):
    display(afl_df.describe(include='all'))

In [ ]:
afl_df.info()

In [ ]:
# Looks further at the 'object' columns for data cleaning
afl_df.select_dtypes(include='object').columns

In [ ]:
afl_df.duplicated().value_counts()

In [ ]:
# Looks like we've got 100 duplicated rows in the dataset - remove these
afl_df = afl_df[afl_df.duplicated()==False]

In [ ]:
afl_df.isna().sum()

In [ ]:
# We've got some missing values - investigate further
with pd.option_context('display.max_columns', None):
    display(afl_df[afl_df.isna().any(axis=1)])

In [ ]:
# Remove these rows
afl_df = afl_df.dropna()
afl_df


In [ ]:
len(afl_df.columns)

In [ ]:
# Lets have a look at the object columns
afl_df.select_dtypes('object').columns

In [ ]:
afl_df[afl_df.select_dtypes('object').columns].head()

In [ ]:
# Look at other column values
display(afl_df['Date'].value_counts())
display(afl_df['Day'].value_counts())
display(afl_df['Kickoff'].value_counts())
display(afl_df['Home Team'].value_counts())
display(afl_df['Away Team'].value_counts())
display(afl_df['Venue'].value_counts())
display(afl_df['Weather Summary'].value_counts())
display(afl_df['Home team injuries'].value_counts())
display(afl_df['Away team injuries'].value_counts())

In [ ]:
# Things we need to fix:
#   - Inconsistencies in team names
#   - Inconsistencies in venue names
#   - Inconsistencies in weather summary
#   - Inconsistencies in injury reporting

In [ ]:
# First, let's see if the Home team always plays at home
# Lets check the realtionship between home teams and venues
afl_df.groupby('Home Team')['Venue'].value_counts()

In [ ]:
afl_df.plot.scatter(x='Home Team', y='Venue', rot=90)

In [ ]:
# So we can drop the Venue
del(afl_df['Venue'])

In [ ]:
# Notice that some teams are given inconsistent naming; correct for this. Also the weather summary has the same problem: use the Ttile case (Capital letter for first letter and lower case for rest of word)

In [ ]:
replace_dict_teams = {
    'Pt. Sovereign Pirates': 'Port Sovereign Pirates',
    'Desert Plains Dingos': 'Desert Plains Dingoes'
}


afl_df[['Home Team', 'Away Team']]  = afl_df[['Home Team', 'Away Team']].replace(replace_dict_teams)
afl_df['Weather Summary'] = afl_df['Weather Summary'].str.title()

In [ ]:
# For Team injuries type variables, set all the False scenarios to 0 and all the True scenarios to 1. Binarise your variable/s.

In [ ]:
true_values = ['TRUE', 'YES', 'T', 'Y']
false_values = ['FALSE', 'NO', 'F', 'N']

afl_df.loc[afl_df['Home team injuries'].str.upper().isin(true_values), 'Home team injuries'] = 1
afl_df.loc[afl_df['Home team injuries'].str.upper().isin(false_values), 'Home team injuries'] = 0
afl_df.loc[afl_df['Away team injuries'].str.upper().isin(true_values), 'Away team injuries'] = 1
afl_df.loc[afl_df['Away team injuries'].str.upper().isin(false_values), 'Away team injuries'] = 0

In [ ]:
# Recheck values
display(afl_df['Date'].value_counts())
display(afl_df['Day'].value_counts())
display(afl_df['Kickoff'].value_counts())
display(afl_df['Home Team'].value_counts())
display(afl_df['Away Team'].value_counts())
display(afl_df['Weather Summary'].value_counts())
display(afl_df['Home team injuries'].value_counts())
display(afl_df['Away team injuries'].value_counts())

In [ ]:
# Convert the date column from string
afl_df['Date'] = pd.to_datetime(afl_df['Date'])

In [ ]:
# Let's extract the time of each game
# We'll use a 'dummy' date of 1/1/2000 whilst retaining the time component to allow the use of datetime functions
afl_df['Time'] = pd.to_datetime(afl_df['Kickoff']).apply(lambda t: t.replace(year=2000, month=1, day=1))
del(afl_df['Kickoff'])

In [ ]:
afl_df['Time']

In [ ]:
_ = afl_df.plot(subplots=True, layout=(20,4), figsize=(20,100))

In [ ]:
# Things to look at:
# outliers in pollen count, temperature, rain
# negative values in totals

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,10))
plt.subplot(1,3,1)
afl_df[['Pollen count',]].boxplot()
plt.subplot(1,3,2)
afl_df[['Temperature (°C)',]].boxplot()
plt.subplot(1, 3,3)
afl_df[['Rain (mm)',]].boxplot()
plt.show()

In [ ]:
# We'll replace these outliers with the means
temp_mean = afl_df[afl_df['Temperature (°C)'] <= 100]['Temperature (°C)'].mean()
afl_df.loc[afl_df['Temperature (°C)'] > 100,'Temperature (°C)'] = temp_mean


In [ ]:
pollen_count_mean = afl_df[afl_df['Pollen count'] <= 20]['Pollen count'].mean()
afl_df.loc[afl_df['Pollen count'] > 20,'Pollen count'] = pollen_count_mean


In [ ]:
rain_mean = afl_df[afl_df['Rain (mm)'] <= 15]['Rain (mm)'].mean()
afl_df.loc[afl_df['Rain (mm)'] > 15,'Rain (mm)'] = rain_mean


In [ ]:
plt.figure(figsize=(10,10))
plt.subplot(1,3,1)
afl_df[['Pollen count',]].boxplot()
plt.subplot(1,3,2)
afl_df[['Temperature (°C)',]].boxplot()
plt.subplot(1, 3,3)
afl_df[['Rain (mm)',]].boxplot()
plt.show()

In [ ]:
# Let's save our cleaned data
from pathlib import Path

dest_dir = Path(data_path / "Processed")
dest_dir.mkdir(parents=True, exist_ok=True)

file_to_write = dest_dir / "afl_fantasy_league_test_data-cleaned.csv"
afl_df.to_csv(file_to_write, index=False)